# Tren en liten språkmodell fra bunnen

**Du skal kunne:** følge tegn fra tekst til embedding, kausal attention og
prediksjon av neste tegn. Vi trener en liten transformer på et kunstig, norsk
meldingsspråk om nett og signaler. Det lille datasettet har få setningsmønstre;
modellen lærer å etterligne dem, ikke fagkunnskap eller vanlig norsk generelt.

Dette er trening fra tilfeldige vekter. Ingen ferdigtrent språkmodell eller
skytjeneste brukes. GPU anbefales for flere treningssteg; små forsøk går på CPU.
Les delene rolig: denne øvelsen bygger på både treningsløkken og attention-forsøket.


## Gjør arbeidsbenken klar

`import` henter verktøy; `as` gir et kortere navn. `Path` håndterer filstier.
Vi finner kursmappen og legger `startkode` i Pythons søkesti, slik at samme kode
virker fra VS Code og JupyterLab. `sett_seed(42)` gir et fast startpunkt for
slumptall; det gjør sammenligning lettere, men lover ikke identiske tall på ulike
maskiner. `velg_enhet()` velger NVIDIA-GPU når CUDA er tilgjengelig, ellers CPU.
En **enhet** er stedet beregningene skjer. Modell og data må ligge på samme enhet.

**Før kjøring:** Gjett om utskriften blir `cpu` eller `cuda`.
**Etter kjøring:** Kontroller at det stemmer med installasjonssjekken.


In [ ]:
from pathlib import Path
import sys
import torch
import matplotlib.pyplot as plt

# Finn kursmappen både fra en notatbok og fra prosjektroten.
her = Path.cwd().resolve()
kandidater = [her, *her.parents, her / "05-dypere-maskinlaering"]
kursmappe = next(p for p in kandidater if (p / "startkode" / "kursverktoy.py").exists())
if str(kursmappe / "startkode") not in sys.path:
    sys.path.insert(0, str(kursmappe / "startkode"))
from kursverktoy import sett_seed, velg_enhet
sett_seed(42)
enhet = velg_enhet()
print("PyTorch:", torch.__version__, "| beregner på:", enhet)


## Del tekstene før vi lager treningsvinduer

Vi kombinerer steder, utstyr, hendelser og tidspunkt til korte meldinger.
`product` lager alle kombinasjonene. `shuffle` stokker hele meldingene med fast seed.
Vi deler **meldinger**, ikke overlappende vinduer, slik at samme melding ikke havner
i både trening og validering. Testdelen er lagt til side. De enkle språkmønstrene
deles likevel mellom delene; dette måler nye kombinasjoner innen et kunstig språk.

`join` setter sammen tekst. Et **vokabular** er listen over tillatte symboler.
Vi erklærer alfabetet på forhånd. `enumerate` gir hvert tegn et heltallsnummer.
En token er her ett tegn; større språkmodeller bruker ofte ordfragmenter.


In [ ]:
from itertools import product
import random
steder = ["oslo", "bergen", "tromsø", "bodø", "stavanger", "trondheim"]
utstyr = ["ruteren", "antennen", "senderen", "kabelen"]
hendelser = ["virker normalt", "må undersøkes", "er byttet ut", "har en feil"]
tider = ["mandag", "tirsdag", "onsdag", "torsdag", "fredag"]
meldinger = [f"i {sted} meldes det at {ting} {hendelse} på {tid}.\n"
             for sted, ting, hendelse, tid in product(steder, utstyr, hendelser, tider)]
random.Random(42).shuffle(meldinger)
n = len(meldinger)
tren_tekst = "".join(meldinger[:int(n*0.8)])
val_tekst = "".join(meldinger[int(n*0.8):int(n*0.9)])
test_tekst = "".join(meldinger[int(n*0.9):])  # spar denne til valgene er tatt
alfabet = sorted(set("abcdefghijklmnopqrstuvwxyzæøå .\n"))
til_id = {tegn: i for i, tegn in enumerate(alfabet)}
def kod(tekst):
    return torch.tensor([til_id[tegn] for tegn in tekst], dtype=torch.long)
def dekod(tall):
    return "".join(alfabet[int(i)] for i in tall)
tren_data, val_data = kod(tren_tekst), kod(val_tekst)
print("Meldinger:", n, "vokabular:", len(alfabet), "treningstegn:", len(tren_data))
print(tren_tekst[:240])


## Input og fasit er forskjøvet ett tegn

Kontekstlengde er hvor mange tidligere tegn modellen får bruke. En batch har form
`[batch, tid]`. Vi velger tilfeldige startsteder i **én** del av dataene om gangen.
`stack` samler utsnittene. Fasit flyttes én posisjon: For input «nett» kan målet
være «ett ». Modellen får trening på alle posisjonene samtidig.

**Stopp:** Les de to utskriftene og pek på forskyvningen.


In [ ]:
kontekst = 64
batchstorrelse = 32

def hent_batch(data, batch=batchstorrelse):
    starter = torch.randint(0, len(data) - kontekst, (batch,))
    x = torch.stack([data[i:i+kontekst] for i in starter])
    y = torch.stack([data[i+1:i+kontekst+1] for i in starter])
    return x.to(enhet), y.to(enhet)

x, y = hent_batch(tren_data)
print("Input:", repr(dekod(x[0].cpu())))
print("Fasit:", repr(dekod(y[0].cpu())))


## Bygg én transformerblokk

En **klasse** er en oppskrift på et objekt. `__init__` lager lagene; `forward`
beskriver hvordan data går gjennom dem. Ved å arve fra `nn.Module` registrerer
PyTorch vektene slik at optimizer finner dem. `self` viser til denne modellen.

`LayerNorm` normaliserer verdier innen hver tegnrepresentasjon. `MultiheadAttention`
lager Q, K og V og flere parallelle attention-hoder. Med bredde 64 og fire hoder
har hvert hode bredde 16. `batch_first=True` betyr `[batch, tid, bredde]`.
Samme normaliserte input brukes til query, key og value: **self-attention**.

`attn_mask=True` betyr at den forbindelsen **blokkeres** i dette API-et.
`need_weights=False` unngår å returnere kart som ikke brukes i treningen.
En **residualforbindelse**, `x + ...`, beholder en direkte vei fra forrige lag.
MLP-delen bearbeider hver posisjon separat etter at attention har blandet kontekst.


In [ ]:
from torch import nn

class Blokk(nn.Module):
    def __init__(self, bredde=64, hoder=4):
        super().__init__()
        self.norm1 = nn.LayerNorm(bredde)
        self.attention = nn.MultiheadAttention(bredde, hoder, dropout=0.0, batch_first=True)
        self.norm2 = nn.LayerNorm(bredde)
        self.mlp = nn.Sequential(nn.Linear(bredde, 4*bredde), nn.ReLU(), nn.Linear(4*bredde, bredde))

    def forward(self, x):
        tid = x.shape[1]
        maske = torch.triu(torch.ones(tid, tid, dtype=torch.bool, device=x.device), diagonal=1)
        normalisert = self.norm1(x)
        blanding, _ = self.attention(normalisert, normalisert, normalisert,
                                     attn_mask=maske, need_weights=False)
        x = x + blanding
        return x + self.mlp(self.norm2(x))


## Fra tegnnummer til logits

`Embedding` er en lærbar oppslagstabell: heltallsnummer inn, vektor ut. Vi legger
sammen tegnembedding og posisjonsembedding; dermed kan modellen skille første tegn
fra femte tegn. To blokker bearbeider dette før et siste lineært lag lager én logit
per mulig neste tegn. Output har form `[batch, tid, vokabular]`.

Dette er en liten transformer med kausal attention. Oppsettet er laget for læring
og utelater mange valg og optimaliseringer i moderne store språkmodeller.


In [ ]:
class LitenSprakmodell(nn.Module):
    def __init__(self, vokabular, bredde=64):
        super().__init__()
        self.tegn = nn.Embedding(vokabular, bredde)
        self.posisjon = nn.Embedding(kontekst, bredde)
        self.blokker = nn.Sequential(Blokk(bredde), Blokk(bredde))
        self.norm = nn.LayerNorm(bredde)
        self.utgang = nn.Linear(bredde, vokabular)

    def forward(self, token):
        pos = torch.arange(token.shape[1], device=token.device)
        x = self.tegn(token) + self.posisjon(pos)
        return self.utgang(self.norm(self.blokker(x)))

sett_seed(42)
modell = LitenSprakmodell(len(alfabet)).to(enhet)
print("Output:", modell(x).shape)
print("Parametre:", sum(p.numel() for p in modell.parameters()))


## Generer tekst før trening

For hvert nytt tegn tar vi logits fra **siste posisjon**, deler på temperatur og
bruker softmax. `multinomial` trekker et tegn fra fordelingen; det gir variasjon.
Lav temperatur konsentrerer trekningen, høy temperatur sprer den mer.
`cat` legger det nye tegnet til sekvensen. Vi gir modellen bare de siste 64 tegnene.
`eval` og `no_grad` gjør dette til inferens, ikke trening.

**Forutsi:** Vil et utrent nett skrive leselige norske meldinger?


In [ ]:
@torch.no_grad()
def generer(modell, start="i ", antall=160, temperatur=0.8):
    if temperatur <= 0:
        raise ValueError("Temperatur må være positiv")
    modell.eval()
    sekvens = kod(start).unsqueeze(0).to(enhet)
    for _ in range(antall):
        logits = modell(sekvens[:, -kontekst:])[:, -1, :]
        neste = torch.multinomial((logits / temperatur).softmax(-1), num_samples=1)
        sekvens = torch.cat([sekvens, neste], dim=1)
    return dekod(sekvens[0].cpu())

print(generer(modell))


## Tren og mål på faste valideringsvinduer

Vi flater batch- og tidsaksen til én akse før cross entropy: hver tegnposisjon er
ett klassifiseringseksempel. `reshape(-1, ...)` lar Python beregne nødvendig lengde.
`clip_grad_norm_` begrenser samlet gradientnorm til 1 for å dempe svært store steg;
det er ikke det samme som å endre læringsraten.

Valideringsvinduene lages én gang med en egen generator. Dermed sammenligner vi de
samme vinduene hver gang uten å påvirke tilfeldig utvalg av treningsvinduer.
Vi lagrer beste vekter på CPU. **300 steg** betyr 300 batcher, ikke 300 epoker.

**Før kjøring:** Skriv ned hva du forventer av tekst før og etter trening.


In [ ]:
generator = torch.Generator().manual_seed(123)
val_starter = torch.randint(0, len(val_data) - kontekst, (64,), generator=generator)
val_x = torch.stack([val_data[i:i+kontekst] for i in val_starter]).to(enhet)
val_y = torch.stack([val_data[i+1:i+kontekst+1] for i in val_starter]).to(enhet)
optimizer = torch.optim.AdamW(modell.parameters(), lr=0.001)
steg_antall = 300  # prøv 1000–3000 etter at det lille forsøket fungerer
logg, beste, beste_feil = [], None, float("inf")
for steg in range(1, steg_antall + 1):
    modell.train()
    x, y = hent_batch(tren_data)
    optimizer.zero_grad()
    logits = modell(x)
    feil = nn.functional.cross_entropy(logits.reshape(-1, len(alfabet)), y.reshape(-1))
    feil.backward()
    nn.utils.clip_grad_norm_(modell.parameters(), max_norm=1.0)
    optimizer.step()
    if steg == 1 or steg % 50 == 0 or steg == steg_antall:
        modell.eval()
        with torch.no_grad():
            val_logits = modell(val_x)
            val_feil = nn.functional.cross_entropy(
                val_logits.reshape(-1, len(alfabet)), val_y.reshape(-1)).item()
        logg.append((steg, feil.item(), val_feil))
        print(f"Steg {steg}: treningsbatch={feil.item():.3f}, validering={val_feil:.3f}")
        if val_feil < beste_feil:
            beste_feil = val_feil
            beste = {k: v.detach().cpu().clone() for k, v in modell.state_dict().items()}
modell.load_state_dict(beste)


## Sammenlign tekst og læringskurve

Treningskurven viser én tilfeldig batch ved hvert loggpunkt, derfor kan den være
ujevn. Valideringskurven bruker de samme 64 vinduene. `perplexity = exp(cross entropy)`
er et mål på hvor usikker modellen er om neste token i gjennomsnitt. Det er ikke
et mål på om teksten er sann, og tall for ulike tokeniseringer kan ikke uten videre
sammenlignes. Her holder vi oss til cross entropy.


In [ ]:
plt.plot([r[0] for r in logg], [r[1] for r in logg], label="En treningsbatch")
plt.plot([r[0] for r in logg], [r[2] for r in logg], label="Faste valideringsvinduer")
plt.xlabel("Treningssteg")
plt.ylabel("Cross entropy per tegn")
plt.legend()
plt.show()
print(generer(modell))


## Spar og bruk modellen

Vi lagrer vekter, alfabet og kontekst. For å bruke filen senere må vi også ha
klassedefinisjonene ovenfor og samme bredde. Dette er et checkpoint til inferens.
`state_dict` inkluderer ikke optimizer, datakilde eller Python-koden.


In [ ]:
resultatmappe = kursmappe / "resultater"
resultatmappe.mkdir(exist_ok=True)
torch.save({"modell": {k: v.cpu() for k, v in modell.state_dict().items()},
            "alfabet": alfabet, "kontekst": kontekst, "bredde": 64},
           resultatmappe / "liten-sprakmodell.pt")


## Undersøk én ting om gangen

Prøv temperatur 0,3 og 1,2 på **samme trente modell**. Deretter kan du trene en ny
modell med flere steg. For endring av kontekst må du bygge modellen på nytt fordi
posisjonstabellen får en annen størrelse. Attention-matrisen vokser med kvadratet
av kontekstlengden; større kontekst kan derfor gi betydelig mer minnebruk.

Når innstillingene er valgt, kan du kode `test_tekst`, lage faste vinduer og måle
cross entropy én gang med samme fremgangsmåte som valideringen. Hold dette adskilt
fra valg av temperatur og antall steg. Testen sier bare noe om det kunstige språket.

**Refleksjon:** Modellen har ingen belønning, bare neste-tegn-fasit. Hvordan skiller
det seg fra reinforcement learning? Kan en korrekt formet melding fortsatt være
faglig feil? Ja: treningsspråket er oppdiktet og gir ingen fakta om virkelige nett.

**Neste:** [08-reinforcement-learning.ipynb](08-reinforcement-learning.ipynb).
